# PW_Bioacoustics Demo

End-to-end walkthrough of the PW_Bioacoustics pipeline using real bird recordings from the
[PteroSet](https://zenodo.org/records/19137071) dataset.

## 0. Setup

Set up the Python environment, verify the working directory, and configure all output paths.

In [ ]:
%matplotlib inline

import os
import sys
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import soundfile as sf
import torchaudio
import torch

warnings.filterwarnings("ignore")

# ── Verify working directory ────────────────────────────────────────────
DEMO_DIR = Path(os.getcwd())
if DEMO_DIR.name != "demo":
    raise RuntimeError(
        f"Run this notebook from PW_Bioacoustics/demo/. Currently in: {DEMO_DIR}\n"
        "  cd PW_Bioacoustics/demo && jupyter notebook bioacoustics_demo.ipynb"
    )

PW_BIO_DIR     = DEMO_DIR.parent        # PW_Bioacoustics/
CAMERATRAP_DIR = PW_BIO_DIR.parent      # CameraTraps/

for p in [str(PW_BIO_DIR), str(CAMERATRAP_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Directory layout ────────────────────────────────────────────
DATA_DIR       = DEMO_DIR / "data"
AUDIOS_DIR     = DATA_DIR / "audios"
LABELS_DIR     = DATA_DIR / "labels"
OUTPUT_DIR     = DEMO_DIR / "output"
SPEC_DIR       = OUTPUT_DIR / "spectrograms"   # shared by both modes
BINARY_DIR     = OUTPUT_DIR / "binary"
MULTICLASS_DIR = OUTPUT_DIR / "multiclass"

for d in [AUDIOS_DIR, LABELS_DIR, SPEC_DIR, BINARY_DIR, MULTICLASS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment ready")
print(f"  PW_Bioacoustics : {PW_BIO_DIR}")
print(f"  Data            : {DATA_DIR}")
print(f"  Output          : {OUTPUT_DIR}")

## 1. Data Exploration

Load metadata for the five PteroSet recordings (project PPA4, Puerto Asís, Putumayo, Colombia).
Each `.Table.1.selections.txt` file is a Raven Pro annotation table with `Begin Time (s)`,
`End Time (s)`, and species `Determination` columns.

**Expected output:** a summary table with file duration, sample rate, and annotation counts.

In [ ]:
PPA4_FILES = [
    "G021_timelapse_20250623",
    "G021_timelapse_20250622",
    "G040_timelapse_20250625",
    "G040_timelapse_20250629",
    "G010_timelapse_20250629",
]

rows = []
for stem in PPA4_FILES:
    txt_path = LABELS_DIR / f"{stem}.Table.1.selections.txt"
    wav_path = AUDIOS_DIR / f"{stem}.wav"
    df = pd.read_csv(txt_path, delimiter="\t")
    info = sf.info(str(wav_path))
    with_species = df["Determination"].apply(
        lambda x: isinstance(x, str) and x.strip() != ""
    ).sum()
    rows.append({
        "file": stem,
        "duration (min)": round(info.duration / 60, 1),
        "sample_rate (kHz)": info.samplerate // 1000,
        "total_annotations": len(df),
        "with_species_id": int(with_species),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nTotal annotations : {summary['total_annotations'].sum()}")
print(f"Total duration    : {summary['duration (min)'].sum():.0f} min")

## 2. Inference

Run inference on all five recordings using our pre-trained model for birds binary classification (`MD_AudioBirds_V1`). This section does **not** require training — it demonstrates how to
apply our model to new audio data.

1. Download the pre-trained `MD_AudioBirds_V1.onnx` model from Zenodo and set up the
inference output directories. The model file is cached locally and skipped on subsequent runs.

    **Expected output:** confirmation that the model file exists or has been downloaded.

In [ ]:
import urllib.request
from PytorchWildlife.data.bioacoustics.bioacoustics_windows import build_inference_windows
from PytorchWildlife.data.bioacoustics.bioacoustics_spectrograms import compute_mel_spectrograms_gpu
from PytorchWildlife.data.bioacoustics.bioacoustics_datasets import BioacousticsInferenceDataset
from torch.utils.data import DataLoader

MODEL_URL  = "https://zenodo.org/records/18177050/files/MD_AudioBirds_V1.onnx?download=1"
INFER_DIR  = OUTPUT_DIR / "inference"
INFER_SPEC = INFER_DIR / "spectrograms"
INFER_DIR.mkdir(parents=True, exist_ok=True)
INFER_SPEC.mkdir(parents=True, exist_ok=True)

MODEL_PATH = INFER_DIR / "MD_AudioBirds_V1.onnx"
if not MODEL_PATH.exists():
    print("Downloading MD_AudioBirds_V1.onnx from Zenodo...")
    urllib.request.urlretrieve(MODEL_URL, str(MODEL_PATH))
    print(f"Model saved to: {MODEL_PATH}")
else:
    print(f"Model already cached: {MODEL_PATH}")

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
SR      = 48000
WIN_SEC = 5.0
OVL_SEC = 4.0
print(f"Device: {DEVICE}")

2. Build sliding inference windows over all five audio files and compute their mel spectrograms.
Windows already computed on disk are skipped automatically.

    **Expected output:** window count per file, spectrogram progress bars, and confirmation of saved files.

In [ ]:
all_wav_files = [str(AUDIOS_DIR / f"{stem}.wav") for stem in PPA4_FILES]

# Build sliding windows for all 5 files
infer_windows = build_inference_windows(
    audios_source=all_wav_files,
    window_size_sec=WIN_SEC,
    overlap_sec=OVL_SEC,
    sample_rate=SR,
)
print(f"Total inference windows: {len(infer_windows)}")

# Compute mel spectrograms (skips existing files automatically)
compute_mel_spectrograms_gpu(
    windows=infer_windows,
    sample_rate=SR,
    n_fft=2048, hop_length=512, n_mels=224, top_db=80.0,
    spectrograms_path=str(INFER_SPEC),
    save_npy=True,
    fill_highfreq=False,
)

# Build DataFrame with full spectrogram paths
infer_df = pd.DataFrame(infer_windows)
infer_df["sound_stem"] = infer_df["sound_path"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)
infer_df["spec_name"] = infer_df.apply(
    lambda r: str(INFER_SPEC / f"{r['sound_stem']}_{r['start']}_{r['end']}.npy"), axis=1
)
print(f"Spectrogram files ready: {len(infer_df)} windows across {infer_df['sound_path'].nunique()} files")

3. Load the ONNX model with `onnxruntime` and run batch inference on all spectrogram windows.

    **Expected output:** model input/output shapes, per-batch progress, and final output shape.

In [ ]:
import onnxruntime as ort

TARGET_SIZE = [224, 469]

infer_ds = BioacousticsInferenceDataset(
    dataframe=infer_df, x_col="spec_name",
    target_size=TARGET_SIZE, normalize=False,
)
infer_dl = DataLoader(infer_ds, batch_size=32, shuffle=False, num_workers=0)

# Load ONNX session
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
             if DEVICE == "cuda" else ["CPUExecutionProvider"])
ort_session = ort.InferenceSession(str(MODEL_PATH), sess_opts, providers=providers)
input_name  = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name

print(f"Input  : {input_name}  shape={ort_session.get_inputs()[0].shape}")
print(f"Outputs: {[o.name for o in ort_session.get_outputs()]}")

# Run inference
all_probs = []
for batch in infer_dl:
    x, _ = batch
    x_np = x.numpy().astype(np.float32)
    logits = ort_session.run([output_name], {input_name: x_np})[0]
    probs = 1.0 / (1.0 + np.exp(-logits.reshape(-1, 1)))
    all_probs.append(probs)

all_probs = np.concatenate(all_probs, axis=0)
print(f"\nInference complete — output shape: {all_probs.shape}")

4. Attach the predicted probabilities to the inference DataFrame, save predictions to CSV,
and display the first few rows.

    **Expected output:** a CSV saved to `output/inference/onnx_predictions.csv` and a table preview.

In [ ]:
infer_df["start_s"] = infer_df["start"] / SR
infer_df["end_s"]   = infer_df["end"]   / SR

if all_probs.shape[1] == 1:
    infer_df["probability"] = all_probs[:, 0]
    infer_df["prediction"]  = (infer_df["probability"] > 0.5).astype(int)
else:
    n_cls = all_probs.shape[1]
    for i in range(n_cls):
        infer_df[f"prob_class_{i}"] = all_probs[:, i]
    infer_df["prediction"] = all_probs.argmax(axis=1)

results_csv = str(INFER_DIR / "onnx_predictions.csv")
infer_df.to_csv(results_csv, index=False)
print(f"Predictions saved to: {results_csv}")
infer_df.head()

5. Visualise the predicted bird-call probability over time for
each of the five audio files. Red shading marks the ground-truth annotation intervals.

    **Expected output:** a 5-panel figure — one subplot per recording.

In [ ]:
is_binary_onnx = "probability" in infer_df.columns

fig, axes = plt.subplots(len(PPA4_FILES), 1,
                         figsize=(14, 3 * len(PPA4_FILES)), sharex=False)

for ax, stem in zip(axes, PPA4_FILES):
    wav_path = str(AUDIOS_DIR / f"{stem}.wav")
    file_df  = infer_df[infer_df["sound_path"] == wav_path].copy()
    t_mid    = (file_df["start_s"] + file_df["end_s"]) / 2

    if is_binary_onnx:
        ax.plot(t_mid, file_df["probability"], color="#1976D2", lw=0.8)
        ax.axhline(0.5, color="grey", lw=0.8, ls="--", label="threshold")
        ax.set_ylabel("P(bird)")
    else:
        prob_cols = [c for c in infer_df.columns if c.startswith("prob_class_")]
        for i, col in enumerate(prob_cols[1:], start=1):
            ax.plot(t_mid, file_df[col], lw=0.8, label=f"class {i}")
        ax.legend(fontsize=7, ncol=3)
        ax.set_ylabel("Probability")

    # Ground-truth annotation spans
    txt_path = LABELS_DIR / f"{stem}.Table.1.selections.txt"
    if txt_path.exists():
        gt = pd.read_csv(txt_path, delimiter="\t")
        for _, row in gt.iterrows():
            ax.axvspan(row["Begin Time (s)"], row["End Time (s)"],
                       alpha=0.18, color="red")

    ax.set_ylim(-0.05, 1.05)
    ax.set_title(stem)
    ax.set_xlabel("Time (s)")

plt.suptitle(
    "ONNX Inference — MD_AudioBirds_V1  (red = ground-truth annotations)",
    y=1.01, fontsize=11,
)
plt.tight_layout()
plt.show()

## 3. Train

### Build COCO Annotations

`PteroSetReader` follows the same `BaseReader` pattern as `data_reader.py` in the source dataset.
It converts Raven Pro selection tables to the COCO-like JSON that `build_windows()` expects.

- **Binary mode** — every annotated event → `category_id=1` (`AVEVOC`)
- **Multiclass mode** — only the top-4 species → `category_id` 1–4; unlabeled events are skipped
  and will fall into background windows during `build_windows()`

In [ ]:
from PytorchWildlife.data.bioacoustics.bioacoustics_annotations import BaseReader, AnnotationCreator

class PteroSetReader(BaseReader):
    """Converts PteroSet Raven Pro annotations to COCO-like JSON.

    Parameters
    ----------
    demo_data_dir : str or Path
        Path to demo/data/  (must contain audios/ and labels/ sub-directories).
    mode : {"binary", "multiclass"}
        - binary     : every AVEVOC event -> category_id=1
        - multiclass : top-4 species (CACCEL, CYAVIO, PSAANG, RAMCAR) -> category_id 1-4
    """

    TOP_SPECIES = ["CACCEL", "CYAVIO", "PSAANG", "RAMCAR"]  # category_id 1-4

    def __init__(self, demo_data_dir, mode="binary"):
        super().__init__(str(demo_data_dir))
        if mode not in ("binary", "multiclass"):
            raise ValueError("mode must be 'binary' or 'multiclass'")
        self.mode = mode
        self.audio_dir  = Path(demo_data_dir) / "audios"
        self.labels_dir = Path(demo_data_dir) / "labels"
        self.output_path = str(Path(demo_data_dir) / f"{mode}_annotations.json")

    def add_dataset_info(self):
        self.annotation_creator.add_info(
            title="PteroSet — demo subset (5 recordings)",
            license="CC BY 4.0",
            description=(
                "5 recordings from project PPA4 (Puerto Asís, Putumayo, Colombia). "
                "Annotated with Raven Pro. Part of the PteroSet dataset "
                "(https://zenodo.org/records/19137071)."
            ),
        )

    def add_sounds(self):
        wav_files = sorted(f for f in os.listdir(self.audio_dir) if f.endswith(".wav"))
        for sound_id, filename in enumerate(wav_files):
            file_path = str(self.audio_dir / filename)
            duration, sample_rate = self.annotation_creator._get_duration_and_sample_rate(file_path)
            self.annotation_creator.add_sound(
                id=sound_id,
                file_name_path=file_path,
                duration=duration,
                sample_rate=sample_rate,
                latitude=float("nan"),
                longitude=float("nan"),
            )

    def add_categories(self):
        if self.mode == "binary":
            cats = [
                {"id": 0, "name": "noise",  "supercategory": "background"},
                {"id": 1, "name": "AVEVOC", "supercategory": "BIO"},
            ]
        else:
            cats = [{"id": 0, "name": "noise", "supercategory": "background"}]
            for i, sp in enumerate(self.TOP_SPECIES, start=1):
                cats.append({"id": i, "name": sp, "supercategory": "AVEVOC"})
        self.annotation_creator.data["categories"] = cats

    def add_annotations(self):
        wav_files = sorted(f for f in os.listdir(self.audio_dir) if f.endswith(".wav"))
        anno_id = 0
        for sound_id, wav_file in enumerate(wav_files):
            stem     = os.path.splitext(wav_file)[0]
            txt_path = self.labels_dir / f"{stem}.Table.1.selections.txt"
            if not txt_path.exists():
                continue
            df = pd.read_csv(txt_path, delimiter="\t")
            for _, row in df.iterrows():
                t_min = float(row["Begin Time (s)"])
                t_max = float(row["End Time (s)"])
                determination = str(row.get("Determination", "")).strip()
                if self.mode == "binary":
                    self.annotation_creator.add_annotation(
                        anno_id=anno_id, sound_id=sound_id,
                        category_id=1, category="AVEVOC",
                        supercategory="BIO", t_min=t_min, t_max=t_max,
                    )
                    anno_id += 1
                else:
                    if determination in self.TOP_SPECIES:
                        cat_id = self.TOP_SPECIES.index(determination) + 1
                        self.annotation_creator.add_annotation(
                            anno_id=anno_id, sound_id=sound_id,
                            category_id=cat_id, category=determination,
                            supercategory="AVEVOC", t_min=t_min, t_max=t_max,
                        )
                        anno_id += 1

Run `PteroSetReader` in both `binary` and `multiclass` modes to produce two COCO-like
JSON annotation files in `data/`.

**Expected output:** paths and annotation counts for `binary_annotations.json` and `multiclass_annotations.json`.

In [ ]:
for mode in ("binary", "multiclass"):
    reader = PteroSetReader(DATA_DIR, mode=mode)
    reader.process_dataset()
    print()

### Binary Classification

**Task:** detect any bird vocalization (`AVEVOC`) vs. background noise.

Pipeline steps:
1. Write config to `pteroset_binary.yaml` and load it
2. `build_windows()` with `strategy="sliding"` and `multiclass=False` → label ∈ {0, 1}
3. `compute_mel_spectrograms_gpu()` → `.npy` files
4. Create train / val / test splits
5. Train `ResNetClassifier` with `num_classes=2`

Write the binary classification config to YAML and load it as a typed config object.
All hyperparameters (window size, spectrogram, training) are stored here for reproducibility.

**Expected output:** printed config summary (sample rate, window size, backbone, etc.).

In [ ]:
import yaml
from PytorchWildlife.data.bioacoustics.bioacoustics_configs import load_config

BINARY_CONFIG_PATH = str(DEMO_DIR / "config" / "pteroset_binary.yaml")
os.makedirs(os.path.dirname(BINARY_CONFIG_PATH), exist_ok=True)

binary_cfg_dict = {
    "name": "pteroset_binary",
    "datasets": ["audios"],
    "class_names": {0: "noise", 1: "AVEVOC"},
    "paths": {
        "data_root":        str(DATA_DIR),
        "output_root":      str(BINARY_DIR),
        "spectrograms_dir": str(SPEC_DIR),
        "annotations_file": "binary_annotations.json",
    },
    "audio": {
        "sample_rate":          48000,
        "window_size_sec":      5.0,
        "overlap_sec":          4.0,
        "window_strategy":      "sliding",
        "multiclass":           False,
    },
    "spectrogram": {
        "n_fft":        2048,
        "hop_length":   512,
        "n_mels":       128,
        "top_db":       80.0,
        "fill_highfreq": False,
        "noise_db_std": 3.0,
        "storage_dtype": "float32",
    },
    "training": {
        "batch_size":     16,
        "num_workers":    2,
        "lr":             1e-4,
        "weight_decay":   1e-4,
        "epochs":         10,
        "backbone":       "resnet18",
        "num_classes":    2,
        "target_size":    [128, 465],
        "normalize":      True,
        "use_specaug":    True,
        "pos_weight":     2.0,
        "conf_threshold": 0.5,
    },
    "splits": {
        "test_size":    0.2,
        "val_size":     0.2,
        "n_splits":     3,
        "random_state": 42,
    },
}

with open(BINARY_CONFIG_PATH, "w") as f:
    yaml.dump(binary_cfg_dict, f, default_flow_style=False, sort_keys=False)

binary_cfg = load_config(BINARY_CONFIG_PATH)
print(f"Config loaded: {binary_cfg.name}")
print(f"  sample_rate    : {binary_cfg.audio.sample_rate} Hz")
print(f"  window_size    : {binary_cfg.audio.window_size_sec}s")
print(f"  strategy       : {binary_cfg.audio.window_strategy}")
print(f"  num_classes    : {binary_cfg.training.num_classes}")
print(f"  backbone       : {binary_cfg.training.backbone}")

Print dataset statistics (annotation counts, class balance) and build the sliding window
list from the binary annotation JSON.

**Expected output:** per-file annotation stats and total window count with label distribution.

In [ ]:
import sys
from prepare_dataset import run_stats, run_windows

run_stats(binary_cfg)
binary_windows = run_windows(binary_cfg)

Compute mel spectrograms for all binary windows. Windows are named
`{sound_filename}_{start}_{end}.npy` and stored in the shared `spectrograms/` directory.
Existing files are skipped automatically.

**Expected output:** per-file progress bars and a count of new vs. cached spectrograms.

In [ ]:
from PytorchWildlife.data.bioacoustics.bioacoustics_spectrograms import compute_mel_spectrograms_gpu

with open(binary_cfg.paths.annotations_path) as f:
    anno_data = json.load(f)
sounds_map = {s["id"]: s for s in anno_data["sounds"]}

inf_windows = [
    {
        "window_id":  w["window_id"],
        "sound_path": sounds_map[w["sound_id"]]["file_name_path"],
        "start":      w["start"],
        "end":        w["end"],
    }
    for w in binary_windows
    if w["sound_id"] in sounds_map
]

compute_mel_spectrograms_gpu(
    windows=inf_windows,
    sample_rate=binary_cfg.audio.sample_rate,
    n_fft=binary_cfg.spectrogram.n_fft,
    hop_length=binary_cfg.spectrogram.hop_length,
    n_mels=binary_cfg.spectrogram.n_mels,
    top_db=binary_cfg.spectrogram.top_db,
    spectrograms_path=str(SPEC_DIR),
    save_npy=True,
    fill_highfreq=binary_cfg.spectrogram.fill_highfreq,
    noise_db_std=binary_cfg.spectrogram.noise_db_std,
    storage_dtype=binary_cfg.spectrogram.storage_dtype,
)
print(f"Spectrograms saved to: {SPEC_DIR}")

Split windows into train / val / test sets using `GroupShuffleSplit` (files stay together)
and `StratifiedGroupKFold` (class balance preserved). CSVs are saved to `output/binary/`.

**Expected output:** split sizes and per-split label distributions.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold


def create_splits(windows, spectrograms_dir, sounds_list, output_dir, cfg):
    """Build train/val/test CSVs with spec_name = {sound_filename}_{start}_{end}.npy."""
    sounds_stems = {
        s["id"]: os.path.splitext(os.path.basename(s["file_name_path"]))[0]
        for s in sounds_list
    }
    df = pd.DataFrame(windows)
    df["sound_filename"] = df["sound_id"].map(sounds_stems)
    df["spec_name"] = df.apply(
        lambda r: f"{r['sound_filename']}_{r['start']}_{r['end']}.npy", axis=1
    )
    df["spec_exists"] = df["spec_name"].apply(
        lambda x: os.path.exists(os.path.join(spectrograms_dir, x))
    )
    df = df[df["spec_exists"]].drop(columns=["spec_exists"])
    print(f"Windows with existing spectrograms: {len(df)}")

    gss = GroupShuffleSplit(
        n_splits=1, test_size=cfg.splits.test_size,
        random_state=cfg.splits.random_state
    )
    trainval_idx, test_idx = next(gss.split(df, df["label"], groups=df["sound_id"]))
    trainval_df = df.iloc[trainval_idx].copy()
    test_df     = df.iloc[test_idx].copy()

    sgkf = StratifiedGroupKFold(
        n_splits=cfg.splits.n_splits, shuffle=True,
        random_state=cfg.splits.random_state
    )
    train_idx, val_idx = next(
        sgkf.split(trainval_df, trainval_df["label"], trainval_df["sound_id"])
    )
    train_df = trainval_df.iloc[train_idx].copy()
    val_df   = trainval_df.iloc[val_idx].copy()

    os.makedirs(output_dir, exist_ok=True)
    train_df.to_csv(os.path.join(output_dir, "train_split.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir,   "val_split.csv"),   index=False)
    test_df.to_csv(os.path.join(output_dir,  "test_split.csv"),  index=False)

    print(f"\nSplit sizes  (train={len(train_df)}  val={len(val_df)}  test={len(test_df)})")
    for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"  {name}: label dist = {split['label'].value_counts().to_dict()}")
    return train_df, val_df, test_df


binary_train, binary_val, binary_test = create_splits(
    binary_windows, str(SPEC_DIR), anno_data["sounds"],
    str(BINARY_DIR), binary_cfg
)

Instantiate `ResNetClassifier` in binary mode (2 classes, BCEWithLogitsLoss) and train with
PyTorch Lightning. The best checkpoint (highest `val/f1`) is saved to `output/binary/checkpoints/`.

**Expected output:** training progress bars, per-epoch metrics, and final test results.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

from PytorchWildlife.models.bioacoustics import ResNetClassifier
from train import SpectrogramDataModule, DataModuleConfig

pl.seed_everything(42)

binary_dm_cfg = DataModuleConfig(
    train_csv=str(BINARY_DIR / "train_split.csv"),
    val_csv  =str(BINARY_DIR / "val_split.csv"),
    test_csv =str(BINARY_DIR / "test_split.csv"),
    root     =str(SPEC_DIR),
    target_size=binary_cfg.training.target_size,
    batch_size =binary_cfg.training.batch_size,
    num_workers=binary_cfg.training.num_workers,
    use_specaug=binary_cfg.training.use_specaug,
    normalize  =binary_cfg.training.normalize,
    num_classes=None,
    use_mixup  =True,
    pin_memory =False,
)
binary_dm = SpectrogramDataModule(binary_dm_cfg)
binary_dm.setup()

binary_model = ResNetClassifier(
    num_classes     =binary_dm.num_classes,
    in_channels     =binary_dm.in_channels,
    backbone        =binary_cfg.training.backbone,
    lr              =binary_cfg.training.lr,
    weight_decay    =binary_cfg.training.weight_decay,
    T_max           =binary_cfg.training.epochs,
    batch_size      =binary_cfg.training.batch_size,
    pos_weight      =binary_cfg.training.pos_weight,
    conf_threshold  =binary_cfg.training.conf_threshold,
    class_names     =list(binary_cfg.class_names.values()),
)
print(f"Mode     : {'Binary' if binary_dm.is_binary else 'Multiclass'}")
print(f"Classes  : {binary_dm.num_classes}")
print(f"Channels : {binary_dm.in_channels}")

binary_logger = CSVLogger(str(BINARY_DIR), name="logs")
binary_ckpt   = ModelCheckpoint(
    monitor="val/f1", mode="max", save_top_k=1,
    dirpath=str(BINARY_DIR / "checkpoints"),
    filename="binary-{epoch:02d}-{val/f1:.3f}",
)
binary_trainer = pl.Trainer(
    max_epochs        =binary_cfg.training.epochs,
    accelerator       ="auto",
    devices           =1,
    precision         ="32",
    gradient_clip_val =1.0,
    log_every_n_steps =5,
    callbacks         =[binary_ckpt, LearningRateMonitor()],
    logger            =binary_logger,
    enable_progress_bar=True,
)

binary_trainer.fit(binary_model, datamodule=binary_dm)
binary_trainer.test(binary_model, datamodule=binary_dm, ckpt_path="best")
print(f"\nBest checkpoint : {binary_ckpt.best_model_path}")
print(f"Best val/f1     : {binary_ckpt.best_model_score:.4f}")

Plot loss and validation F1 curves from the CSVLogger output to inspect training dynamics.
`IPython.display` is used explicitly to ensure the figure renders inline after the trainer output.

**Expected output:** two side-by-side plots — train/val loss and val F1 vs. epoch.

In [ ]:
from IPython.display import display as ipy_display

metrics_path = Path(binary_logger.log_dir) / "metrics.csv"
metrics = pd.read_csv(metrics_path)

train_metrics = metrics[metrics["train/loss"].notna()].copy()
val_metrics   = metrics[metrics["val/loss"].notna()].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(train_metrics["epoch"], train_metrics["train/loss"], label="train")
ax1.plot(val_metrics["epoch"],   val_metrics["val/loss"],     label="val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Binary — Loss")
ax1.legend()

ax2.plot(val_metrics["epoch"], val_metrics["val/f1"], color="#1976D2")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1")
ax2.set_title("Binary — Validation F1")

plt.tight_layout()
ipy_display(fig)
plt.close(fig)

### Multiclass Classification

Count annotation occurrences per species across all five recordings and derive the
`TOP_SPECIES` list automatically from the data (the N most-annotated species).
A bar chart highlights which species are used in the multiclass demo.

**Expected output:** a horizontal bar chart of the top-10 species and printed counts.

In [ ]:
species_counts = {}
for stem in PPA4_FILES:
    df = pd.read_csv(LABELS_DIR / f"{stem}.Table.1.selections.txt", delimiter="\t")
    for det in df["Determination"].dropna():
        det = str(det).strip()
        if det:
            species_counts[det] = species_counts.get(det, 0) + 1

# Derive TOP_SPECIES automatically from the most frequent species in the data
N_TOP = 4
top_all = sorted(species_counts.items(), key=lambda x: -x[1])
TOP_SPECIES = [sp for sp, _ in top_all[:N_TOP]]

top10 = top_all[:10]
labels_plot = [sp for sp, _ in top10]
counts_plot = [c for _, c in top10]
colors = ["#1976D2" if sp in TOP_SPECIES else "#B0BEC5" for sp in labels_plot]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels_plot[::-1], counts_plot[::-1], color=colors[::-1])
ax.set_xlabel("Annotation count (across 5 recordings)")
ax.set_title(f"Top 10 species — blue bars = top-{N_TOP} species used for multiclass demo")
patch_blue = mpatches.Patch(color="#1976D2", label=f"Top-{N_TOP} multiclass species")
patch_grey = mpatches.Patch(color="#B0BEC5", label="Other / background")
ax.legend(handles=[patch_blue, patch_grey], fontsize=8)
plt.tight_layout()
plt.show()

print(f"TOP_SPECIES (derived from data): {TOP_SPECIES}")
print(f"Top-{N_TOP} species counts: { {sp: species_counts.get(sp, 0) for sp in TOP_SPECIES} }")
total_pos = sum(species_counts.values())
print(f"All annotated events (binary positives): {total_pos}")

**Task:** classify windows into one of 5 categories — `noise`, `CACCEL`, `CYAVIO`, `PSAANG`, `RAMCAR`.

Key differences from binary:
- `build_windows(multiclass=True)` → `label = category_id` (0–4) instead of 0/1
- `ResNetClassifier(num_classes=5)` → CrossEntropyLoss + softmax instead of BCEWithLogitsLoss
- Spectrograms are **shared** — only new windows (if any) are computed

Write the multiclass config to YAML and load it. The key differences vs. the binary config are
`multiclass=True` in the audio section and `num_classes=5` in training.

**Expected output:** printed config summary confirming 5 classes and multiclass mode.

In [ ]:
MULTICLASS_CONFIG_PATH = str(DEMO_DIR / "config" / "pteroset_multiclass.yaml")

multiclass_cfg_dict = {
    "name": "pteroset_multiclass",
    "datasets": ["audios"],
    "class_names": {0: "noise", 1: "CACCEL", 2: "CYAVIO", 3: "PSAANG", 4: "RAMCAR"},
    "paths": {
        "data_root":        str(DATA_DIR),
        "output_root":      str(MULTICLASS_DIR),
        "spectrograms_dir": str(SPEC_DIR),
        "annotations_file": "multiclass_annotations.json",
    },
    "audio": {
        "sample_rate":         48000,
        "window_size_sec":     5.0,
        "overlap_sec":         4.0,
        "window_strategy":     "sliding",
        "multiclass":          True,
    },
    "spectrogram": {
        "n_fft":         2048,
        "hop_length":    512,
        "n_mels":        128,
        "top_db":        80.0,
        "fill_highfreq": False,
        "noise_db_std":  3.0,
        "storage_dtype": "float32",
    },
    "training": {
        "batch_size":     16,
        "num_workers":    2,
        "lr":             1e-4,
        "weight_decay":   1e-4,
        "epochs":         10,
        "backbone":       "resnet18",
        "num_classes":    5,
        "target_size":    [128, 465],
        "normalize":      True,
        "use_specaug":    True,
    },
    "splits": {
        "test_size":    0.2,
        "val_size":     0.2,
        "n_splits":     3,
        "random_state": 42,
    },
}

with open(MULTICLASS_CONFIG_PATH, "w") as f:
    yaml.dump(multiclass_cfg_dict, f, default_flow_style=False, sort_keys=False)

multiclass_cfg = load_config(MULTICLASS_CONFIG_PATH)
print(f"Config loaded: {multiclass_cfg.name}")
print(f"  num_classes : {multiclass_cfg.training.num_classes}")
print(f"  multiclass  : {multiclass_cfg.audio.multiclass}")

Build the multiclass sliding windows. Only annotations for the top-4 species receive a
positive label; all other windows are background (label=0).

**Expected output:** total window count and label distribution across the 5 classes.

In [ ]:
multiclass_windows = run_windows(multiclass_cfg)
print(f"\nLabel distribution: {pd.Series([w['label'] for w in multiclass_windows]).value_counts().to_dict()}")

Compute spectrograms for multiclass windows. Because the same 5-second window definition
is shared with the binary task, most files already exist and are skipped.

**Expected output:** count of new vs. cached spectrograms.

In [ ]:
with open(multiclass_cfg.paths.annotations_path) as f:
    mc_anno_data = json.load(f)
mc_sounds_map = {s["id"]: s for s in mc_anno_data["sounds"]}

mc_inf_windows = [
    {
        "window_id":  w["window_id"],
        "sound_path": mc_sounds_map[w["sound_id"]]["file_name_path"],
        "start":      w["start"],
        "end":        w["end"],
    }
    for w in multiclass_windows
    if w["sound_id"] in mc_sounds_map
]

compute_mel_spectrograms_gpu(
    windows=mc_inf_windows,
    sample_rate=multiclass_cfg.audio.sample_rate,
    n_fft=multiclass_cfg.spectrogram.n_fft,
    hop_length=multiclass_cfg.spectrogram.hop_length,
    n_mels=multiclass_cfg.spectrogram.n_mels,
    top_db=multiclass_cfg.spectrogram.top_db,
    spectrograms_path=str(SPEC_DIR),
    save_npy=True,
    fill_highfreq=multiclass_cfg.spectrogram.fill_highfreq,
    noise_db_std=multiclass_cfg.spectrogram.noise_db_std,
    storage_dtype=multiclass_cfg.spectrogram.storage_dtype,
)

Split multiclass windows into train / val / test, reusing the same `create_splits` helper.

**Expected output:** split sizes and per-split label distributions for all 5 classes.

In [ ]:
multiclass_train, multiclass_val, multiclass_test = create_splits(
    multiclass_windows, str(SPEC_DIR), mc_anno_data["sounds"],
    str(MULTICLASS_DIR), multiclass_cfg
)

Instantiate `ResNetClassifier` in multiclass mode (5 classes, CrossEntropyLoss) and train.
MixUp is disabled for multiclass to avoid mixing soft labels across 5 categories.

**Expected output:** training progress bars, per-epoch metrics, and final test results.

In [ ]:
pl.seed_everything(42)

mc_dm_cfg = DataModuleConfig(
    train_csv  =str(MULTICLASS_DIR / "train_split.csv"),
    val_csv    =str(MULTICLASS_DIR / "val_split.csv"),
    test_csv   =str(MULTICLASS_DIR / "test_split.csv"),
    root       =str(SPEC_DIR),
    target_size=multiclass_cfg.training.target_size,
    batch_size =multiclass_cfg.training.batch_size,
    num_workers=multiclass_cfg.training.num_workers,
    use_specaug=multiclass_cfg.training.use_specaug,
    normalize  =multiclass_cfg.training.normalize,
    num_classes=multiclass_cfg.training.num_classes,
    use_mixup  =False,
    pin_memory =False,
)
mc_dm = SpectrogramDataModule(mc_dm_cfg)
mc_dm.setup()

mc_model = ResNetClassifier(
    num_classes  =multiclass_cfg.training.num_classes,
    in_channels  =mc_dm.in_channels,
    backbone     =multiclass_cfg.training.backbone,
    lr           =multiclass_cfg.training.lr,
    weight_decay =multiclass_cfg.training.weight_decay,
    T_max        =multiclass_cfg.training.epochs,
    batch_size   =multiclass_cfg.training.batch_size,
    class_names  =list(multiclass_cfg.class_names.values()),
)
print(f"Mode    : {'Binary' if mc_dm.is_binary else 'Multiclass'}")
print(f"Classes : {mc_dm.num_classes}")

mc_logger = CSVLogger(str(MULTICLASS_DIR), name="logs")
mc_ckpt   = ModelCheckpoint(
    monitor="val/f1", mode="max", save_top_k=1,
    dirpath=str(MULTICLASS_DIR / "checkpoints"),
    filename="multiclass-{epoch:02d}-{val/f1:.3f}",
)
mc_trainer = pl.Trainer(
    max_epochs        =multiclass_cfg.training.epochs,
    accelerator       ="auto",
    devices           =1,
    precision         ="32",
    gradient_clip_val =1.0,
    log_every_n_steps =5,
    callbacks         =[mc_ckpt, LearningRateMonitor()],
    logger            =mc_logger,
    enable_progress_bar=True,
)

mc_trainer.fit(mc_model, datamodule=mc_dm)
mc_trainer.test(mc_model, datamodule=mc_dm, ckpt_path="best")
print(f"\nBest checkpoint : {mc_ckpt.best_model_path}")
print(f"Best val/f1     : {mc_ckpt.best_model_score:.4f}")

Plot loss and macro-F1 curves from the multiclass CSVLogger.
`IPython.display` is used explicitly to ensure the figure renders inline after the trainer output.

**Expected output:** two side-by-side plots — train/val loss and val macro-F1 vs. epoch.

In [ ]:
mc_metrics = pd.read_csv(Path(mc_logger.log_dir) / "metrics.csv")
mc_train   = mc_metrics[mc_metrics["train/loss"].notna()].copy()
mc_val     = mc_metrics[mc_metrics["val/loss"].notna()].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(mc_train["epoch"], mc_train["train/loss"], label="train")
ax1.plot(mc_val["epoch"],   mc_val["val/loss"],     label="val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Multiclass — Loss")
ax1.legend()

ax2.plot(mc_val["epoch"], mc_val["val/f1"], color="#388E3C")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1 (macro)")
ax2.set_title("Multiclass — Validation F1")

plt.tight_layout()
ipy_display(fig)
plt.close(fig)